In [11]:
import simpy
import shapely.geometry
import pandas as pd
import openclsim.core as core
import openclsim.model as model

# --------------------------------------------------
# 1. Initialise simpy environment and registry
# --------------------------------------------------
my_env = simpy.Environment(initial_time=0)
registry = {}

# --------------------------------------------------
# 2. Geometry
# --------------------------------------------------
quay_loc = shapely.geometry.Point(4.18055556, 52.18664444)
vessel_loc = shapely.geometry.Point(4.18055556, 52.18664444)
yard_loc = shapely.geometry.Point(4.25222222, 52.11428333)

# --------------------------------------------------
# 3. Define object classes
# --------------------------------------------------
Site = type(
    "Site",
    (
        core.Identifiable,
        core.Log,
        core.Locatable,
        core.HasContainer,
        core.HasResource,
    ),
    {},
)

VesselUnloader = type(
    "VesselUnloader",
    (
        core.Identifiable,
        core.ContainerDependentMovable,
        core.HasResource,
        core.Processor,
        core.LoadingFunction,
        core.UnloadingFunction,
    ),
    {},
)

LandTransporter = type(
    "LandTransporter",
    (
        core.Identifiable,
        core.ContainerDependentMovable,
        core.Processor,
        core.HasResource,
        core.LoadingFunction,
        core.UnloadingFunction,
    ),
    {},
)

# --------------------------------------------------
# 4. Create fixed objects
# --------------------------------------------------
quay_site = Site(
    env=my_env,
    name="Quay Site",
    geometry=quay_loc,
    capacity=1000,
    level=0,
)

yard_block = Site(
    env=my_env,
    name="YardBlockA",
    geometry=yard_loc,
    capacity=1000,
    level=0,
)

initial_teu = 50

vessel01 = VesselUnloader(
    env=my_env,
    name="vessel01",
    geometry=vessel_loc,
    loading_rate=1,
    unloading_rate=1,
    capacity=initial_teu,
    level=initial_teu,
    compute_v=lambda x: 10,
)

qc01 = VesselUnloader(
    env=my_env,
    name="qc_01",
    geometry=quay_loc,
    loading_rate=1,
    unloading_rate=1,
    capacity=1,
    level=0,
    compute_v=lambda x: 10,
)

# --------------------------------------------------
# 5. Parameters
# --------------------------------------------------
fleet_size = 10
qc_cycle_time = 240
pm_load_time = 120
pm_unload_time = 120
pm_speed = 5

# --------------------------------------------------
# 6. Create PM fleet
# --------------------------------------------------
pm_fleet = []
for i in range(1, fleet_size + 1):
    pm = LandTransporter(
        env=my_env,
        name=f"pm_{i:02d}",
        geometry=quay_loc,
        loading_rate=1,
        unloading_rate=1,
        capacity=1,
        level=0,
        compute_v=lambda x, v=pm_speed: v,
    )
    pm_fleet.append(pm)

# --------------------------------------------------
# 7. Create QC discharge sequence
# --------------------------------------------------
qc_steps = []
for n in range(1, initial_teu + 1):
    qc_step = model.ShiftAmountActivity(
        env=my_env,
        name=f"QC01_discharge_step_{n}",
        registry=registry,
        processor=qc01,
        origin=vessel01,
        destination=quay_site,
        amount=1,
        duration=qc_cycle_time,
        start_event=[] if n == 1 else [
            {"name": f"QC01_discharge_step_{n-1}", "type": "activity", "state": "done"}
        ],
    )
    qc_steps.append(qc_step)

qc_process = model.SequentialActivity(
    env=my_env,
    name="QC_discharge_sequence",
    registry=registry,
    sub_processes=qc_steps,
)

# --------------------------------------------------
# 8. Create PM jobs with proper per-PM chaining
# --------------------------------------------------
pm_jobs = []
pm_load_activities = []
pm_travel_activities = []
pm_unload_activities = []
pm_return_activities = []

last_job_name_per_pm = {pm.name: None for pm in pm_fleet}

for n in range(1, initial_teu + 1):
    mover = pm_fleet[(n - 1) % fleet_size]

    load_start_events = [
        {"name": f"QC01_discharge_step_{n}", "type": "activity", "state": "done"}
    ]

    if last_job_name_per_pm[mover.name] is not None:
        load_start_events.append(
            {"name": last_job_name_per_pm[mover.name], "type": "activity", "state": "done"}
        )

    lm_load = model.ShiftAmountActivity(
        env=my_env,
        name=f"TEU_{n}_load_{mover.name}",
        registry=registry,
        processor=mover,
        origin=quay_site,
        destination=mover,
        amount=1,
        duration=pm_load_time,
        start_event=load_start_events,
    )

    lm_travel_to_yard = model.MoveActivity(
        env=my_env,
        name=f"TEU_{n}_to_yard_{mover.name}",
        registry=registry,
        mover=mover,
        destination=yard_block,
        start_event=[
            {"name": f"TEU_{n}_load_{mover.name}", "type": "activity", "state": "done"}
        ],
    )

    lm_unload = model.ShiftAmountActivity(
        env=my_env,
        name=f"TEU_{n}_unload_{mover.name}",
        registry=registry,
        processor=mover,
        origin=mover,
        destination=yard_block,
        amount=1,
        duration=pm_unload_time,
        start_event=[
            {"name": f"TEU_{n}_to_yard_{mover.name}", "type": "activity", "state": "done"}
        ],
    )

    lm_return = model.MoveActivity(
        env=my_env,
        name=f"TEU_{n}_return_{mover.name}",
        registry=registry,
        mover=mover,
        destination=quay_site,
        start_event=[
            {"name": f"TEU_{n}_unload_{mover.name}", "type": "activity", "state": "done"}
        ],
    )

    job = model.SequentialActivity(
        env=my_env,
        name=f"TEU_{n}_job_{mover.name}",
        registry=registry,
        sub_processes=[lm_load, lm_travel_to_yard, lm_unload, lm_return],
    )

    pm_jobs.append(job)
    pm_load_activities.append(lm_load)
    pm_travel_activities.append(lm_travel_to_yard)
    pm_unload_activities.append(lm_unload)
    pm_return_activities.append(lm_return)

    last_job_name_per_pm[mover.name] = f"TEU_{n}_job_{mover.name}"

# --------------------------------------------------
# 9. Combine system
# --------------------------------------------------
system = model.ParallelActivity(
    env=my_env,
    name="terminal_fleet_system",
    registry=registry,
    sub_processes=[qc_process, *pm_jobs],
)

# --------------------------------------------------
# 10. Register and run
# --------------------------------------------------
model.register_processes([system])

for pm in pm_fleet:
    print(f"{pm.name} initial location: {pm.geometry}")

my_env.run()

# --------------------------------------------------
# 11. Inspect results
# --------------------------------------------------
print("Vessel:", vessel01.container.get_level())
print("Quay:", quay_site.container.get_level())
print("Yard:", yard_block.container.get_level())

for pm in pm_fleet:
    print(f"{pm.name} final location: {pm.geometry}")

# --------------------------------------------------
# 12. Collect logs seperately for each object
# --------------------------------------------------
all_logs = []
for obj in [vessel01, quay_site, yard_block, qc01] + pm_fleet:
    if hasattr(obj, "logbook"):
        df = pd.DataFrame(obj.logbook)
        if not df.empty:
            df["object"] = obj.name
            all_logs.append(df)

if all_logs:
    logs_df = pd.concat(all_logs, ignore_index=True)
    print(logs_df.head())
else:
    print("No logs found.")

#export logs to csv for further analysis
logs_df.to_csv("activity_logs.csv", index=False)
import pandas as pd
import re

# --------------------------------------------------
# 12A. Build detailed ordered event log
# --------------------------------------------------
def parse_activity_metadata(activity_name):
    if activity_name.startswith("QC01_discharge_step_"):
        m = re.search(r"QC01_discharge_step_(\d+)", activity_name)
        teu_id = int(m.group(1)) if m else None
        return teu_id, None, "qc_discharge", 1

    m = re.search(r"TEU_(\d+)_(load|to_yard|unload|return)_(pm_\d+)", activity_name)
    if m:
        teu_id = int(m.group(1))
        stage = m.group(2)
        pm_name = m.group(3)

        stage_map = {
            "load": ("pm_load", 2),
            "to_yard": ("pm_travel_to_yard", 3),
            "unload": ("pm_unload", 4),
            "return": ("pm_return", 5),
        }

        activity_type, activity_order = stage_map[stage]
        return teu_id, pm_name, activity_type, activity_order

    return None, None, "unknown", 999


activity_frames = []
all_activities = (
    qc_steps
    + pm_load_activities
    + pm_travel_activities
    + pm_unload_activities
    + pm_return_activities
)

for act in all_activities:
    if hasattr(act, "log"):
        df = pd.DataFrame(act.log)
        if not df.empty:
            teu_id, pm_name, activity_type, activity_order = parse_activity_metadata(act.name)
            df = df.copy()
            df["activity_name"] = act.name
            df["teu_id"] = teu_id
            df["pm_name"] = pm_name
            df["activity_type"] = activity_type
            df["activity_order"] = activity_order
            activity_frames.append(df)

if activity_frames:
    activity_logs_df = pd.concat(activity_frames, ignore_index=True)
else:
    activity_logs_df = pd.DataFrame()

activity_logs_df = activity_logs_df.drop(
    columns=[c for c in ["ActivityID", "ObjectState", "ActivityLabel"] if c in activity_logs_df.columns],
    errors="ignore"
)

activity_logs_df = activity_logs_df[
    activity_logs_df["ActivityState"].isin(["WAIT_START", "WAIT_STOP", "START", "STOP"])
].copy()

activity_logs_df = activity_logs_df[activity_logs_df["teu_id"].notna()].copy()

activity_logs_df = activity_logs_df.sort_values(
    by=["teu_id", "activity_order", "Timestamp"]
).reset_index(drop=True)

print(activity_logs_df.head(50))
activity_logs_df.to_csv("activity_sequence_logs.csv", index=False)

# --------------------------------------------------
# 12B. Build compact one-row-per-TEU summary
# --------------------------------------------------
summary_rows = []

for teu_id in sorted(activity_logs_df["teu_id"].unique()):
    teu_df = activity_logs_df[activity_logs_df["teu_id"] == teu_id]

    row = {
        "teu_id": teu_id,
        "pm_name": teu_df["pm_name"].dropna().iloc[0] if teu_df["pm_name"].notna().any() else None,
    }

    for activity_type in ["qc_discharge", "pm_load", "pm_travel_to_yard", "pm_unload", "pm_return"]:
        part = teu_df[teu_df["activity_type"] == activity_type]

        start_rows = part[part["ActivityState"] == "START"]
        stop_rows = part[part["ActivityState"] == "STOP"]
        wait_start_rows = part[part["ActivityState"] == "WAIT_START"]
        wait_stop_rows = part[part["ActivityState"] == "WAIT_STOP"]

        row[f"{activity_type}_wait_start"] = wait_start_rows["Timestamp"].iloc[0] if not wait_start_rows.empty else None
        row[f"{activity_type}_wait_stop"] = wait_stop_rows["Timestamp"].iloc[0] if not wait_stop_rows.empty else None
        row[f"{activity_type}_start"] = start_rows["Timestamp"].iloc[0] if not start_rows.empty else None
        row[f"{activity_type}_stop"] = stop_rows["Timestamp"].iloc[0] if not stop_rows.empty else None

    summary_rows.append(row)

activity_summary_df = pd.DataFrame(summary_rows)
print(activity_summary_df.head(20))
activity_summary_df.to_csv("activity_summary_by_teu.csv", index=False)


pm_01 initial location: POINT (4.18055556 52.18664444)
pm_02 initial location: POINT (4.18055556 52.18664444)
pm_03 initial location: POINT (4.18055556 52.18664444)
pm_04 initial location: POINT (4.18055556 52.18664444)
pm_05 initial location: POINT (4.18055556 52.18664444)
pm_06 initial location: POINT (4.18055556 52.18664444)
pm_07 initial location: POINT (4.18055556 52.18664444)
pm_08 initial location: POINT (4.18055556 52.18664444)
pm_09 initial location: POINT (4.18055556 52.18664444)
pm_10 initial location: POINT (4.18055556 52.18664444)
Vessel: 0
Quay: 0
Yard: 50
pm_01 final location: POINT (4.18055556 52.18664444)
pm_02 final location: POINT (4.18055556 52.18664444)
pm_03 final location: POINT (4.18055556 52.18664444)
pm_04 final location: POINT (4.18055556 52.18664444)
pm_05 final location: POINT (4.18055556 52.18664444)
pm_06 final location: POINT (4.18055556 52.18664444)
pm_07 final location: POINT (4.18055556 52.18664444)
pm_08 final location: POINT (4.18055556 52.18664444)

In [6]:
distance = vessel_loc.distance(yard_loc)
print(f"Distance from vessel to yard: {distance:.4f} degrees")

Distance from vessel to yard: 0.1018 degrees
